## IMPORTING REQUEST & REGISTER THE API KEY

We import two libraries:
- requests: lets Python send messages over the internet to Claude's API
- subprocess: lets Python run terminal commands on our computer

The API_KEY is our password to access Claude. Every request we send must include it.

In [ ]:
import requests
import subprocess
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("API key loaded:", API_KEY[:10] + "...")


## CALLING THE CLAUDE API

This function sends a conversation to Claude and returns its reply.
We send the full messages history every time because Claude has no memory between calls.
We use raw text output (not structured outputs) so we can practice parsing Claude's 
replies ourselves using string handling — this is the core of Part 1.

In [2]:
def ask_claude(messages, system_prompt):
    response = requests.post(
        url="https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json"
        },
        json={
            "model": "claude-haiku-4-5-20251001",
            "max_tokens": 500,
            "system": system_prompt,
            "messages": messages
        }
    )
    return response.json()['content'][0]['text']

## SYSTEM PROMPT

The system prompt gives Claude its instructions before the conversation starts.
It tells Claude it is a Windows agent and must reply in EXACTLY one of two formats:
- COMMAND: <command> when it needs to run something
- ANSWER: <text> when it has a final answer

This is how we do function calling without frameworks — we use plain English 
instructions and then parse the text output ourselves in Python.

In [3]:
SYSTEM_PROMPT = """You are a helpful agent that can run commands on Windows.

You are running on a Windows computer, so always use Windows commands:
- Use 'dir' instead of 'ls'
- Use 'echo %DATE%' to get the date
- Use 'wmic logicaldisk' for disk space

When you need to run a command, reply in EXACTLY this format:
COMMAND: dir

When you have a final answer, reply in EXACTLY this format:
ANSWER: your answer here

Never reply with anything else. Always use one of these two formats."""

## RUN AGENT FUNCTION

run_agent — the ReAct loop (Reason + Act)
1. Send the conversation history to Claude
2. Read Claude's reply
3. If Claude wants to run a command → run it, add result to history, loop again
4. If Claude has a final answer → print it and stop
5. Loop until it hits break



In [4]:
def run_agent(user_input, max_commands=5):
    messages = [{"role": "user", "content": user_input}]
    commands_run = 0

    print(f"User: {user_input}")
    print("---")

    while True:
        reply = ask_claude(messages, SYSTEM_PROMPT)
        print(f"Claude raw reply: {reply}")

        reply_type, content = parse_reply(reply)

        if reply_type == "command":
            if commands_run >= max_commands:
                print(f"Safety limit reached! Max {max_commands} commands allowed.")
                break
            print(f"Running: {content}")
            result = run_command(content)
            print(f"Result: {result}")
            commands_run += 1
            print(f"Commands used: {commands_run}/{max_commands}")
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Command result: {result}"})

        elif reply_type == "answer":
            print(f"Final answer: {content}")
            break

## RUN A COMMAND


Uses subprocess to run a terminal command locally on the computer.
The agent's brain (Claude) runs in the cloud, but the commands execute locally.

To run the command in the terminal and give back it printed

result.stdout — what the command printed normally | 
 result.stderr — what the command printed if there was an error.

In [ ]:
DANGEROUS_COMMANDS = [
    "rm ", "rm -", "del ", "del /", "format ",
    "shutdown", "rmdir", "rd ", "erase", "cipher",
    ":(){", "mkfs", "dd ", "wget", "curl"
]

def run_command(command):
    command_lower = command.lower().strip()
    for blocked in DANGEROUS_COMMANDS:
        if blocked in command_lower:
            return f"BLOCKED: '{blocked}' is not allowed for safety reasons."
    
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout + result.stderr

## COMMAND OR ANSWER

parse_reply function

To look at what Claude said, figure out if it's a command or an answer and extract the thing we want.

Reads Claude's reply and checks if it starts with "COMMAND:" or "ANSWER:".
Returns the type ("command" or "answer") and the actual content.


In [6]:
def parse_reply(reply):
    if reply.startswith("COMMAND:"):
        command = reply.replace("COMMAND: ", "").strip()
        return "command", command
    elif reply.startswith("ANSWER: "):
        answer = reply.replace("ANSWER: ", "").strip()
        return "answer", answer
    else:
        return "unknown", reply

TESTING THE AGENT

In [7]:
run_agent("what files are in my current folder?")

User: what files are in my current folder?
---
Claude raw reply: COMMAND: dir
Running: dir
Result:  Volume in drive C has no label.
 Volume Serial Number is 383E-408F

 Directory of c:\ML 2026\ML-Assignment2\Part1

05/26/2026  04:07 AM    <DIR>          .
05/26/2026  05:44 AM    <DIR>          ..
05/26/2026  04:06 AM               127 .env
05/26/2026  04:06 AM            32,503 P1.ipynb
               2 File(s)         32,630 bytes
               2 Dir(s)  67,869,257,728 bytes free

Commands used: 1/5
Claude raw reply: ANSWER: Your current folder (c:\ML 2026\ML-Assignment2\Part1) contains 2 files:
1. .env (127 bytes)
2. P1.ipynb (32,503 bytes)
Final answer: Your current folder (c:\ML 2026\ML-Assignment2\Part1) contains 2 files:
1. .env (127 bytes)
2. P1.ipynb (32,503 bytes)


In [8]:
run_agent("What is today's date?")

User: What is today's date?
---
Claude raw reply: COMMAND: echo %DATE%
Running: echo %DATE%
Result: Tue 05/26/2026

Commands used: 1/5
Claude raw reply: ANSWER: Today's date is Tuesday, May 26, 2026
Final answer: Today's date is Tuesday, May 26, 2026


In [9]:
run_agent("How much free space do I have on my disk?")

User: How much free space do I have on my disk?
---
Claude raw reply: COMMAND: wmic logicaldisk get name,freespace,size
Running: wmic logicaldisk get name,freespace,size
Result: FreeSpace    Name  Size          

67868377088  C:    510902226944  

64474955776  G:    510902226944  




Commands used: 1/5
Claude raw reply: COMMAND: powershell -Command "$freeC = 67868377088 / 1GB; $freeG = 64474955776 / 1GB; Write-Host \"C: drive free space: $([Math]::Round($freeC, 2)) GB`nG: drive free space: $([Math]::Round($freeG, 2)) GB\""
Running: powershell -Command "$freeC = 67868377088 / 1GB; $freeG = 64474955776 / 1GB; Write-Host \"C: drive free space: $([Math]::Round($freeC, 2)) GB`nG: drive free space: $([Math]::Round($freeG, 2)) GB\""
Result: C: drive free space: 63.21 GB
G: drive free space: 60.05 GB

Commands used: 2/5
Claude raw reply: ANSWER: You have 63.21 GB of free space on your C: drive and 60.05 GB of free space on your G: drive, for a total of 123.26 GB of free disk space.
Final answ

## CHAT LOOP


A simple loop that keeps asking the user for input and runs the agent.
Type 'quit' to exit. This makes the agent feel like a real conversation.

In [10]:
print("Agent ready! Type 'quit' to exit.")
print("---")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        print("Goodbye!")
        break
    
    if user_input.strip() == "":
        continue
    
    run_agent(user_input)
    print("---")

Agent ready! Type 'quit' to exit.
---
User: exit
---
Claude raw reply: ANSWER: I'm ready to help you run commands on Windows. Please provide a command you'd like me to execute, and I'll run it for you using the proper Windows command format.
Final answer: I'm ready to help you run commands on Windows. Please provide a command you'd like me to execute, and I'll run it for you using the proper Windows command format.
---
Goodbye!
